<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Third_Places.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Obtención de POIs

In [24]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.2 MB/s eta 0:00:00


In [25]:
!wget https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
!unzip latinobarometro-2024-stata-v20250817.zip

--2026-05-27 16:19:10--  https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
Resolving www.latinobarometro.org (www.latinobarometro.org)... 80.28.53.190
Connecting to www.latinobarometro.org (www.latinobarometro.org)|80.28.53.190|:443... connected.
HTTP request sent, awaiting response... 200 200
Length: 6691592 (6.4M) [application/x-zip-compressed]
Saving to: ‘latinobarometro-2024-stata-v20250817.zip.1’

latinobarometro-202 100%[===================>]   6.38M  5.74MB/s    in 1.1s    

2026-05-27 16:19:13 (5.74 MB/s) - ‘latinobarometro-2024-stata-v20250817.zip.1’ saved [6691592/6691592]

Archive:  latinobarometro-2024-stata-v20250817.zip
replace Latinobarometro_2024_Stata_eng_v20250817.dta? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [26]:
import pandas as pd

# abrir reader
reader = pd.read_stata(
    "Latinobarometro_2024_Stata_esp_v20250817.dta",
    convert_categoricals=False,
    iterator=True
)

# dataframe
df24 = reader.read()

# labels de variables
variable_labels = reader.variable_labels()

# TODOS los value labels
value_labels = reader.value_labels()

In [27]:
import pandas as pd
import numpy as np
import osmnx as ox

# =========================================================
# LEER CSV CON LAT/LON
# =========================================================

geo = pd.read_csv("ciudades_latlon.csv")

# limpiar nombres
geo["City"] = geo["City"].astype(str).str.strip()

# =========================================================
# ARMAR DF DE CIUDADES ÚNICAS DESDE LATINOBARÓMETRO
# =========================================================

pais_labels = value_labels["IDENPA"]
ciudad_labels = value_labels["CIUDAD"]

cities = (
    df24[["IDENPA", "CIUDAD", "TAMCIUD"]]
    .dropna()
    .drop_duplicates(subset=["IDENPA", "CIUDAD"])
    .copy()
)

# mapear labels
cities["country"] = cities["IDENPA"].map(pais_labels)
cities["city_label"] = cities["CIUDAD"].map(ciudad_labels)

# limpiar país
cities["country"] = (
    cities["country"]
    .astype(str)
    .str.replace(r"\[%\d+%\]\s*", "", regex=True)
    .str.strip()
)

# limpiar ciudad
cities["city_label"] = (
    cities["city_label"]
    .astype(str)
    .str.strip()
)

# =========================================================
# MERGE CON LAT/LON
# =========================================================

cities = cities.merge(
    geo[["City", "Latitude_num", "Longitude_num"]],
    left_on="city_label",
    right_on="City",
    how="left"
)

# renombrar
cities = cities.rename(columns={
    "Latitude_num": "lat",
    "Longitude_num": "lon"
})

# sacar ciudades sin coordenadas
cities = cities.dropna(subset=["lat", "lon"])

# =========================================================
# RADIO SEGÚN TAMAÑO
# =========================================================

radius = 1500

# =========================================================
# TAGS THIRD PLACES
# =========================================================
tags = {

    "amenity": [

        # sociabilidad informal
        "bar",
        "pub",
        "cafe",

        # encuentro comunitario
        "community_centre",
        "social_centre",

        # cultura
        "arts_centre",
        "theatre",
        "music_venue",

        # religiosidad / comunidad
        "place_of_worship",


    ],

}

# =========================================================
# LOOP
# =========================================================

results = []

for _, row in cities.iterrows():

    try:

        city = row["city_label"]
        country = row["country"]

        lat = row["lat"]
        lon = row["lon"]

        tam = int(row["TAMCIUD"])

        #radius = radius_map.get(tam, 5000)

        # descargar POIs
        pois = ox.features_from_point(
            (lat, lon),
            tags=tags,
            dist=radius
        )

        # eliminar duplicados
        pois = pois.drop_duplicates(subset=["geometry"])

        n_places = len(pois)

        # =====================================================
        # POBLACIÓN ESTIMADA SEGÚN TAMCIUD
        # =====================================================

        tam_to_pop = {
            1: 2500,      # <5k
            2: 7500,      # 5k-10k
            3: 15000,     # 10k-20k
            4: 30000,     # 20k-40k
            5: 45000,     # 40k-50k
            6: 75000,     # 50k-100k
            7: 250000,    # 100k+
            8: 2500000    # capitales
        }

        estimated_pop = tam_to_pop.get(tam, 50000)

        # =====================================================
        # THIRD PLACES POR HABITANTE
        # =====================================================

        density = (
            n_places / estimated_pop
        ) * 100000

        results.append({
            "country": country,
            "city": city,
            "tamciud": tam,
            "lat": lat,
            "lon": lon,
            "radius_m": radius,
            "third_places": n_places,
            "third_places_density": density
        })

        print(f"OK: {city} -> {n_places}")

    except Exception as e:

        print(f"ERROR: {city} -> {e}")

# =========================================================
# DATAFRAME FINAL
# =========================================================

df_third_places = pd.DataFrame(results)

print(df_third_places.head())

FileNotFoundError: [Errno 2] No such file or directory: 'ciudades_latlon.csv'

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
from tqdm.auto import tqdm

# =========================================================
# LEER CSV CON LAT/LON
# =========================================================

geo = pd.read_csv("ciudades_latlon.csv")

# limpiar nombres
geo["City"] = geo["City"].astype(str).str.strip()

# =========================================================
# ARMAR DF DE CIUDADES ÚNICAS DESDE LATINOBARÓMETRO
# =========================================================

pais_labels = value_labels["IDENPA"]
ciudad_labels = value_labels["CIUDAD"]

cities = (
    df24[["IDENPA", "CIUDAD", "TAMCIUD"]]
    .dropna()
    .drop_duplicates(subset=["IDENPA", "CIUDAD"])
    .copy()
)

# mapear labels
cities["country"] = cities["IDENPA"].map(pais_labels)
cities["city_label"] = cities["CIUDAD"].map(ciudad_labels)

# limpiar país
cities["country"] = (
    cities["country"]
    .astype(str)
    .str.replace(r"\[%\d+%\]\s*", "", regex=True)
    .str.strip()
)

# limpiar ciudad
cities["city_label"] = (
    cities["city_label"]
    .astype(str)
    .str.strip()
)

# =========================================================
# MERGE CON LAT/LON
# =========================================================

cities = cities.merge(
    geo[["City", "Latitude_num", "Longitude_num"]],
    left_on="city_label",
    right_on="City",
    how="left"
)

# renombrar
cities = cities.rename(columns={
    "Latitude_num": "lat",
    "Longitude_num": "lon"
})

# sacar ciudades sin coordenadas
cities = cities.dropna(subset=["lat", "lon"])


# =========================================================
# CONFIG
# =========================================================

radius = 1500


tags = {

    "amenity": [

        # sociabilidad informal
        "bar",
        "pub",
        "cafe",

        # encuentro comunitario
        "community_centre",
        "social_centre",

        # cultura
        "arts_centre",
        "theatre",
        "music_venue",

        # religiosidad / comunidad
        "place_of_worship",

        # política / militancia / asociaciones
        "social_facility"
    ],

    "office": [

        "political_party",
        "association",
        "ngo"
    ],

    "club": [

        # clubes sociales/deportivos
        "sport",
        "social"
    ],

    "leisure": [

        # espacio público
        "park",
        "garden",
        "common",

        # deporte social
        "sports_centre",
        "pitch"
    ]
}

# =========================================================
# POBLACIÓN ESTIMADA
# =========================================================

tam_to_pop = {
    1: 2500,
    2: 7500,
    3: 15000,
    4: 30000,
    5: 45000,
    6: 75000,
    7: 250000,
    8: 2500000
}

# =========================================================
# LISTA DE TODOS LOS TAGS INDIVIDUALES
# =========================================================

all_tag_values = []

for k, vals in tags.items():
    for v in vals:
        all_tag_values.append((k, v))

# =========================================================
# LOOP
# =========================================================

results = []

cities_subset = cities.iloc[:]

for _, row in tqdm(
    cities_subset.iterrows(),
    total=len(cities_subset)
):

    try:

        city = row["city_label"]
        country = row["country"]

        lat = row["lat"]
        lon = row["lon"]

        tam = int(row["TAMCIUD"])

        estimated_pop = tam_to_pop.get(tam, 50000)

        # =====================================================
        # DESCARGAR TODO JUNTO
        # =====================================================

        pois = ox.features_from_point(
            (lat, lon),
            tags=tags,
            dist=radius
        )

        if len(pois) == 0:
            continue

        pois = pois.drop_duplicates(subset=["geometry"])

        # =====================================================
        # RESULTADO BASE
        # =====================================================

        row_result = {
            "country": country,
            "city": city,
            "tamciud": tam,
            "lat": lat,
            "lon": lon,
            "radius_m": radius
        }

        # =====================================================
        # CONTAR CADA TAG
        # =====================================================

        total_places = 0

        for main_tag, sub_tag in all_tag_values:

            try:

                count = (
                    pois[pois[main_tag] == sub_tag]
                    .shape[0]
                )

            except:
                count = 0

            varname = f"{main_tag}_{sub_tag}"

            row_result[varname] = count

            total_places += count

            # versión per capita
            row_result[f"{varname}_per100k"] = (
                count / estimated_pop
            ) * 100000

        # =====================================================
        # TOTAL
        # =====================================================

        row_result["third_places"] = total_places

        row_result["third_places_per100k"] = (
            total_places / estimated_pop
        ) * 100000

        results.append(row_result)

    except Exception as e:

        print(f"ERROR: {city} -> {e}")

# =========================================================
# DF FINAL
# =========================================================

df_third_places = pd.DataFrame(results)

In [ ]:
df_third_places.columns

In [ ]:
df_third_places.to_csv('third_places.csv')

In [ ]:
from google.colab import files

# Guardar CSV
df_third_places.to_csv('third_places.csv', index=False)

# Descargar
files.download('third_places.csv')

# Comparativa

In [1]:
import pandas as pd


df_third_places = pd.read_csv('https://raw.githubusercontent.com/LCaravaggio/Happiness_Polarization/refs/heads/main/third_places.csv')

In [2]:
!wget https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
!unzip latinobarometro-2024-stata-v20250817.zip

--2026-05-27 16:22:50--  https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
Resolving www.latinobarometro.org (www.latinobarometro.org)... 80.28.53.190
Connecting to www.latinobarometro.org (www.latinobarometro.org)|80.28.53.190|:443... connected.
HTTP request sent, awaiting response... 200 200
Length: 6691592 (6.4M) [application/x-zip-compressed]
Saving to: ‘latinobarometro-2024-stata-v20250817.zip’

latinobarometro-202 100%[===================>]   6.38M  5.45MB/s    in 1.2s    

2026-05-27 16:22:52 (5.45 MB/s) - ‘latinobarometro-2024-stata-v20250817.zip’ saved [6691592/6691592]

Archive:  latinobarometro-2024-stata-v20250817.zip
  inflating: Latinobarometro_2024_Stata_eng_v20250817.dta  
  inflating: Latinobarometro_2024_Stata_esp_v20250817.dta  
  inflating: Latinobarometro_2024_Cuestionario_esp.pdf  
  inflating: Latinobarometro_2024_Cuestionario_eng.pdf  


In [3]:
import pandas as pd

# abrir reader
reader = pd.read_stata(
    "Latinobarometro_2024_Stata_esp_v20250817.dta",
    convert_categoricals=False,
    iterator=True
)

# dataframe
df24 = reader.read()

# labels de variables
variable_labels = reader.variable_labels()

# TODOS los value labels
value_labels = reader.value_labels()

In [5]:
import numpy as np

conversion = {

    4: 1,  # para nada
    3: 2,  # no muy
    2: 3,  # bastante
    1: 4   # muy
}


def convertir(val):

    try:

        num = float(val)

        # missings típicos latinobarómetro
        if num in [97, 98, 99]:
            return np.nan

        return num

    except:

        return np.nan

df24["life_satisfaction"] = (
    df24["P1ST"]
    .map(conversion)
)

df24['engagement'] = (
    df24["P36STGBS"].map(conversion)
)

df24["pol_scale"] = (
    df24["P16ST"]
    .apply(convertir)
)

# distancia al centro
df24["polarization"] = (
    np.abs(df24["pol_scale"] - 5)
)

/tmp/ipykernel_1290/3264565117.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df24["pol_scale"] = (
/tmp/ipykernel_1290/3264565117.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df24["polarization"] = (


In [11]:
import pandas as pd
from scipy.stats import pearsonr

# =========================================================
# COLAPSAR LIFE SATISFACTION A NIVEL CIUDAD
# =========================================================

city_life = (
    df24
    .groupby("CIUDAD", as_index=False)[["life_satisfaction", "engagement", "polarization"]]
    .mean()
)

ciudad_labels = value_labels["CIUDAD"]

city_life["city"] = city_life["CIUDAD"].map(ciudad_labels)

# =========================================================
# MERGE
# =========================================================

df_corr = city_life.merge(
    df_third_places,
    on="city",
    how="inner"
)

# =========================================================
# VARIABLES A CORRELACIONAR
# =========================================================

exclude_cols = [
    "country",
    "city",
    "CIUDAD",
    "lat",
    "lon",
    "engagement",
    "polarization",
    "Unnamed: 0",
    "CIUDAD_x",
    "CIUDAD_y"
]

vars_to_test = [
    c for c in df_corr.columns
    if c not in exclude_cols
]

# dejar solo numéricas
vars_to_test = [
    c for c in vars_to_test
    if pd.api.types.is_numeric_dtype(df_corr[c])
]

# sacar variable dependiente
vars_to_test.remove("life_satisfaction")

# =========================================================
# PEARSON RAW
# =========================================================

results = []

y = df_corr["life_satisfaction"]

for var in vars_to_test:

    x = df_corr[var]

    tmp = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna()

    # evitar variables constantes
    if len(tmp) < 6 or tmp["x"].nunique() <= 1:
        continue

    r, p = pearsonr(tmp["x"], tmp["y"])

    results.append({
        "variable": var,
        "pearson_r": r,
        "p_value": p,
        "n": len(tmp)
    })

# =========================================================
# RESULTADOS
# =========================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("pearson_r", ascending=False)
    .reset_index(drop=True)
)

print("Comparativa con Felicidad: ")
results_df

Comparativa con Felicidad: 


,variable,pearson_r,p_value,n
0,amenity_place_of_worship,0.320851,0.023098,50
1,club_sport,0.309902,0.028514,50
2,leisure_pitch,0.301858,0.033135,50
3,office_political_party,0.264573,0.063349,50
4,third_places,0.222781,0.119930,50
5,amenity_bar,0.214630,0.134449,50
6,office_political_party_per100k,0.204325,0.154643,50
7,amenity_cafe_per100k,0.183625,0.201791,50
8,club_sport_per100k,0.175700,0.222281,50
9,office_association_per100k,0.170583,0.236256,50


In [12]:

# =========================================================
# PEARSON RAW
# =========================================================

results = []

y = df_corr["engagement"]

for var in vars_to_test:

    x = df_corr[var]

    tmp = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna()

    # evitar variables constantes
    if len(tmp) < 6 or tmp["x"].nunique() <= 1:
        continue

    r, p = pearsonr(tmp["x"], tmp["y"])

    results.append({
        "variable": var,
        "pearson_r": r,
        "p_value": p,
        "n": len(tmp)
    })

# =========================================================
# RESULTADOS
# =========================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("pearson_r", ascending=False)
    .reset_index(drop=True)
)


print("Comparativa con Engagement: ")
results_df

Comparativa con Engagement: 


,variable,pearson_r,p_value,n
0,office_association_per100k,0.468021,0.000609,50
1,office_association,0.462931,0.000711,50
2,leisure_park,0.336759,0.016788,50
3,leisure_garden,0.320421,0.023293,50
4,third_places,0.318997,0.023949,50
5,amenity_place_of_worship,0.210783,0.141743,50
6,leisure_garden_per100k,0.204127,0.155054,50
7,amenity_pub,0.199816,0.164150,50
8,amenity_community_centre,0.194426,0.176061,50
9,amenity_pub_per100k,0.191366,0.183098,50


In [13]:
# =========================================================
# PEARSON RAW
# =========================================================

results = []

y = df_corr["polarization"]

for var in vars_to_test:

    x = df_corr[var]

    tmp = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna()

    # evitar variables constantes
    if len(tmp) < 6 or tmp["x"].nunique() <= 1:
        continue

    r, p = pearsonr(tmp["x"], tmp["y"])

    results.append({
        "variable": var,
        "pearson_r": r,
        "p_value": p,
        "n": len(tmp)
    })

# =========================================================
# RESULTADOS
# =========================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("pearson_r", ascending=False)
    .reset_index(drop=True)
)


print("Comparativa con Polarization: ")
results_df

Comparativa con Polarization: 


,variable,pearson_r,p_value,n
0,leisure_pitch_per100k,0.426803,0.001995,50
1,third_places_per100k,0.410024,0.003104,50
2,amenity_place_of_worship_per100k,0.376434,0.007052,50
3,leisure_sports_centre_per100k,0.266155,0.061727,50
4,leisure_garden_per100k,0.250400,0.079453,50
5,leisure_park_per100k,0.242151,0.090212,50
6,office_political_party_per100k,0.233594,0.102541,50
7,office_association_per100k,0.209814,0.143627,50
8,amenity_bar_per100k,0.153887,0.285975,50
9,amenity_community_centre_per100k,0.139962,0.332329,50


# News

In [14]:
df_news = pd.read_csv('https://raw.githubusercontent.com/LCaravaggio/Happiness_Polarization/refs/heads/main/Political%20Violence%20Index.csv')

In [15]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.8 MB/s eta 0:00:00


In [21]:
from scipy.stats import pearsonr
from unidecode import unidecode

# =========================================================
# MAPEAR LABELS
# =========================================================

df24["country_name"] = df24["IDENPA"].map(value_labels["IDENPA"])
df24["city_name"] = df24["CIUDAD"].map(value_labels["CIUDAD"])

# =========================================================
# FUNCION LIMPIEZA
# =========================================================

def clean_text(series):

    s = (
        series.astype(str)
        .apply(unidecode)
        .str.lower()
        .str.strip()
    )

    # sacar [%32%]
    s = s.str.replace(r"\[%\d+%\]", "", regex=True)

    # sacar prefijos tipo "AR:"
    s = s.str.replace(r"^[a-z]{2}:\s*", "", regex=True)

    # reemplazar guiones por espacios
    s = s.str.replace("-", " ", regex=False)

    # sacar puntuacion restante
    s = s.str.replace(r"[^\w\s]", "", regex=True)

    # espacios dobles
    s = s.str.replace(r"\s+", " ", regex=True)

    return s.str.strip()

# =========================================================
# LIMPIAR
# =========================================================

df24["country_clean"] = clean_text(df24["country_name"])
df24["city_clean"] = clean_text(df24["city_name"])

df_news["country_clean"] = clean_text(df_news["country_name"])
df_news["city_clean"] = clean_text(df_news["city_name"])

# =========================================================
# COLAPSAR DF24
# =========================================================

city_stats = (
    df24
    .groupby(
        ["country_clean", "city_clean"],
        as_index=False
    )[["life_satisfaction", "engagement", "polarization"]]
    .mean()
)

# =========================================================
# MERGE
# =========================================================

merged = df_news.merge(
    city_stats,
    on=["country_clean", "city_clean"],
    how="inner"
)

print("Matches:", len(merged))

# =========================================================
# LIFE
# =========================================================

tmp = merged[
    ["news_index_3", "life_satisfaction"]
].dropna()

r_life, p_life = pearsonr(
    tmp["news_index_3"].astype(float),
    tmp["life_satisfaction"].astype(float)
)

# =========================================================
# ENGAGEMENT
# =========================================================

tmp = merged[
    ["news_index_3", "engagement"]
].dropna()

r_eng, p_eng = pearsonr(
    tmp["news_index_3"].astype(float),
    tmp["engagement"].astype(float)
)

# =========================================================
# POLARIZATION
# =========================================================

tmp = merged[
    ["news_index_3", "polarization"]
].dropna()

r_pol, p_pol = pearsonr(
    tmp["news_index_3"].astype(float),
    tmp["polarization"].astype(float)
)


print("\nnews_index_3 vs life_satisfaction")
print("r =", round(r_life, 3), "| p =", round(p_life, 4))

print("\nnews_index_3 vs engagement")
print("r =", round(r_eng, 3), "| p =", round(p_eng, 4))

print("\nnews_index_3 vs polarization")
print("r =", round(r_pol, 3), "| p =", round(p_pol, 4))

Matches: 959

news_index_3 vs life_satisfaction
r = -0.094 | p = 0.0036

news_index_3 vs engagement
r = 0.069 | p = 0.0317

news_index_3 vs polarization
r = -0.018 | p = 0.5841
